In [1]:
# type: ignore


import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

llm_model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
    extra_body={"enable_thinking": False},
)

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: bool = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


structured_llm_grader = llm_model.with_structured_output(GradeHallucinations)

system_prompt = """
You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts.

You MUST output a valid JSON object with ONLY the key "binary_score"."""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader

In [3]:
documents = """Agentic RAG（Agentic Retrieval-Augmented Generation，智能体增强检索生成）
是一种将 自主 AI 智能体（Autonomous Agents）与 检索增强生成（RAG）相结合的先进架构。
它超越了传统 RAG 的“一次检索 + 一次生成”静态流程，赋予系统动态规划、多步推理、自我反思和工具调用等能力。"""

hallucination_grader.invoke(
    {
        "documents": documents,
        "generation": "Agentic RAG 融合自主智能体与 RAG，支持动态推理、多步检索与自我优化。",
    }
)

GradeHallucinations(binary_score=True)

In [6]:
hallucination_grader.invoke(
    {
        "documents": documents,
        "generation": "人工智能是让机器模拟人类智能行为（如学习、推理、感知和决策）的技术与科学。",
    }
)

GradeHallucinations(binary_score=False)